# FBRef Paste Parser

Paste schedule data from FBRef directly into the `RAW_TEXT` variable below.
Run the parse cell and it appends to the master CSV.

**Format expected:** tab-separated rows copied from FBRef schedule pages:
`Wk  Day  Date  Time  Home  Score  Away  Attendance  Venue  Referee  Match Report  Notes`

**What to paste for each season (5 competitions × 5 seasons = ~25 pastes):**
- Premier League
- FA Cup
- League Cup
- UEFA Champions League
- UEFA Europa League
- UEFA Europa Conference League (from 2021-22 onwards)

In [218]:
import pandas as pd
import re
from pathlib import Path
from io import StringIO

OUTPUT_DIR = Path("../../../../infra/data/json/fbref").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_CSV = OUTPUT_DIR / "fixtures_manual.csv"

print(f"Master CSV: {MASTER_CSV}")
if MASTER_CSV.exists():
    existing = pd.read_csv(MASTER_CSV)
    print(f"Existing rows: {len(existing)}")
else:
    print("No file yet — will be created on first parse")

Master CSV: /Users/admin/dev/algobetting/infra/data/json/fbref/fixtures_manual.csv
Existing rows: 8486


## Paste cell

1. Set `COMPETITION` and `SEASON`
2. Paste the copied FBRef table into `RAW_TEXT`
3. Run this cell and the one below

In [219]:
COMPETITION = "Conference League"   # change for each paste
SEASON      = "2021-2022"        # change for each paste

RAW_TEXT = """

Round	Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
Group stage	1	Tue	2021-09-14	17:30 (16:30)	Maccabi Tel Aviv FC il	4–1	am Alashkert	6,934	Bloomfield Stadium	Espen Andreas Eskås	Match Report	
Group stage	1	Tue	2021-09-14	17:30 (16:30)	Maccabi Haifa il	0–0	nl Feyenoord	15,850	Sammy Ofer Stadium	Fabio Maresca	Match Report	
Group stage	1	Thu	2021-09-16	18:45	Slavia Prague cz	3–1	de Union Berlin	15,286	Sinobo Stadium	Fábio Veríssimo	Match Report	
Group stage	1	Thu	2021-09-16	18:45	Rennes fr	2–2	eng Tottenham Hotspur	22,000	Roazhon Park	Halis Özkahya	Match Report	
Group stage	1	Thu	2021-09-16	18:45	Slovan Bratislava sk	1–3	dk FC Copenhagen	9,833	Štadión Tehelné pole	Michael Fabbri	Match Report	
Group stage	1	Thu	2021-09-16	18:45	NŠ Mura si	0–2	nl Vitesse	3,300	Mestni Štadion Fazanerija	Mohammed Al Hakim	Match Report	
Group stage	1	Thu	2021-09-16	18:45	Lincoln Red Imps FC gi	0–2	gr PAOK	668	Victoria Stadium	Robert Harvey	Match Report	
Group stage	1	Thu	2021-09-16	19:45 (18:45)	HJK Helsinki fi	0–2	at LASK	3,612	Telia 5G -areena	Kirill Levnikov	Match Report	
Group stage	1	Thu	2021-09-16	19:45 (18:45)	FC Flora ee	0–1	be Gent	2,666	A. Le Coq Arena	Trustin Farrugia Cann	Match Report	
Group stage	1	Thu	2021-09-16	20:30 (16:30)	FC Kairat kz	0–0	cy AC Omonia	7,038	Ortalıq Stadion	Serhiy Boyko	Match Report	
Group stage	1	Thu	2021-09-16	20:45 (18:45)	Qarabağ az	0–0	ch Basel	17,586	Tofiq Bəhramov adına Respublika stadionu	Erik Lambrechts	Match Report	
Group stage	1	Thu	2021-09-16	21:00	Jablonec cz	1–0	ro CFR Cluj	2,470	Stadion Střelnice	Enea Jorgji	Match Report	
Group stage	1	Thu	2021-09-16	21:00	Bodø/Glimt no	3–1	ua Zorya Luhansk	2,703	Aspmyra Stadion	Dennis Higler	Match Report	
Group stage	1	Thu	2021-09-16	21:00	Roma it	5–1	bg CSKA Sofia	29,876	Stadio Olimpico	Glenn Nyberg	Match Report	
Group stage	1	Thu	2021-09-16	21:00	Randers dk	2–2	nl AZ Alkmaar	3,365	Cepheus Park Randers	Duje Strukan	Match Report	
Group stage	1	Thu	2021-09-16	22:00 (21:00)	Anorthosis cy	0–2	rs Partizan	5,000	Stadio Antonis Papadopoulos	Peter Kjærsgaard-Andersen	Match Report	
Group stage	2	Thu	2021-09-30	18:45	AZ Alkmaar nl	1–0	cz Jablonec	9,071	AFAS Stadion	Rade Obrenović	Match Report	
Group stage	2	Thu	2021-09-30	18:45	Partizan rs	2–0	ee FC Flora	4,845	Stadion Partizana	Julian Weinberger	Match Report	
Group stage	2	Thu	2021-09-30	18:45	LASK at	1–1	il Maccabi Tel Aviv FC	1,000	Wörthersee Stadion	Mykola Balakin	Match Report	
Group stage	2	Thu	2021-09-30	18:45	Gent be	2–0	cy Anorthosis	10,209	GHELAMCO-arena	Filip Glova	Match Report	
Group stage	2	Thu	2021-09-30	19:45 (18:45)	CSKA Sofia bg	0–0	no Bodø/Glimt	7,291	Stadion Vasil Levski	Aleksandrs Anufrijevs	Match Report	
Group stage	2	Thu	2021-09-30	19:45 (18:45)	CFR Cluj ro	1–1	dk Randers	3,825	Stadionul Dr. Constantin Rădulescu	Davide Massa	Match Report	
Group stage	2	Thu	2021-09-30	19:45 (18:45)	Zorya Luhansk ua	0–3	it Roma	7,622	Slavutych-Arena	Luís Godinho	Match Report	
Group stage	2	Thu	2021-09-30	20:00 (21:00)	Tottenham Hotspur eng	5–1	si NŠ Mura	25,121	Tottenham Hotspur Stadium	Igal Frid	Match Report	
Group stage	2	Thu	2021-09-30	20:45 (18:45)	Alashkert am	2–4	fi HJK Helsinki	2,000	Vazgen Sargsyan anvan Hanrapetakan Marza...	Horațiu Fesnic	Match Report	
Group stage	2	Thu	2021-09-30	21:00	Vitesse nl	1–2	fr Rennes	14,571	GelreDome	Kevin Clancy	Match Report	
Group stage	2	Thu	2021-09-30	21:00	FC Copenhagen dk	3–1	gi Lincoln Red Imps FC	13,418	Parken	Kateryna Monzul	Match Report	
Group stage	2	Thu	2021-09-30	21:00	Union Berlin de	3–0	il Maccabi Haifa	23,342	Olympiastadion Berlin	Nick Walsh	Match Report	
Group stage	2	Thu	2021-09-30	21:00	Basel ch	4–2	kz FC Kairat	8,712	St. Jakob-Park	Stuart Attwell	Match Report	
Group stage	2	Thu	2021-09-30	21:00	Feyenoord nl	2–1	cz Slavia Prague	38,505	Stadion Feijenoord	Radu Petrescu	Match Report	
Group stage	2	Thu	2021-09-30	22:00 (21:00)	AC Omonia cy	1–4	az Qarabağ	8,345	Stadio GSP (1902)	Bobby Madden	Match Report	
Group stage	2	Thu	2021-09-30	22:00 (21:00)	PAOK gr	1–1	sk Slovan Bratislava	7,537	Stadio Toumbas	João Pinheiro	Match Report	
Group stage	3	Thu	2021-10-21	17:30 (16:30)	HJK Helsinki fi	0–5	il Maccabi Tel Aviv FC	4,883	Telia 5G -areena	Pavel Orel	Match Report	
Group stage	3	Thu	2021-10-21	18:45	FC Copenhagen dk	1–2	gr PAOK	19,552	Parken	Erik Lambrechts	Match Report	
Group stage	3	Thu	2021-10-21	18:45	Vitesse nl	1–0	eng Tottenham Hotspur	23,931	GelreDome	Harm Osmers	Match Report	
Group stage	3	Thu	2021-10-21	18:45	NŠ Mura si	1–2	fr Rennes	3,500	Ljudski vrt	Volen Chinkov	Match Report	
Group stage	3	Thu	2021-10-21	18:45	Bodø/Glimt no	6–1	it Roma	5,652	Aspmyra Stadion	Ali Palabıyık	Match Report	
Group stage	3	Thu	2021-10-21	18:45	Feyenoord nl	3–1	de Union Berlin	36,100	Stadion Feijenoord	Giorgi Kruashvili	Match Report	
Group stage	3	Thu	2021-10-21	19:45 (18:45)	Anorthosis cy	2–2	ee FC Flora	4,322	Neo GSP	István Vad	Match Report	
Group stage	3	Thu	2021-10-21	19:45 (18:45)	Maccabi Haifa il	1–0	cz Slavia Prague	10,000	Sammy Ofer Stadium	Ádám Farkas	Match Report	
Group stage	3	Thu	2021-10-21	20:45 (18:45)	Alashkert am	0–3	at LASK	1,500	Vazgen Sargsyan anvan Hanrapetakan Marza...	Jens Maae	Match Report	
Group stage	3	Thu	2021-10-21	20:45 (18:45)	Qarabağ az	2–1	kz FC Kairat	17,252	Tofiq Bəhramov adına Respublika stadionu	Yevhen Aranovskyi	Match Report	
Group stage	3	Thu	2021-10-21	21:00	Basel ch	3–1	cy AC Omonia	10,056	St. Jakob-Park	Stéphanie Frappart	Match Report	
Group stage	3	Thu	2021-10-21	21:00	Jablonec cz	2–2	dk Randers	3,678	Stadion Střelnice	Chang Ssu-yu	Match Report	
Group stage	3	Thu	2021-10-21	21:00	Partizan rs	0–1	be Gent	8,943	Stadion Partizana	Bobby Madden	Match Report	
Group stage	3	Thu	2021-10-21	21:00	Slovan Bratislava sk	2–0	gi Lincoln Red Imps FC	6,108	Štadión Tehelné pole	Vilhjálmur Thórarinsson	Match Report	
Group stage	3	Thu	2021-10-21	22:00 (21:00)	CFR Cluj ro	0–1	nl AZ Alkmaar	3,628	Stadionul Dr. Constantin Rădulescu	Halis Özkahya	Match Report	
Group stage	3	Thu	2021-10-21	22:00 (21:00)	CSKA Sofia bg	0–1	ua Zorya Luhansk	3,158	Stadion Vasil Levski	Manuel Schuettengruber	Match Report	
Group stage	4	Thu	2021-11-04	17:30 (16:30)	FC Flora ee	2–2	cy Anorthosis	2,023	A. Le Coq Arena	Kári Høvdanum	Match Report	
Group stage	4	Thu	2021-11-04	18:45	LASK at	2–0	am Alashkert	400	Wörthersee Stadion	Mohammed Al Hakim	Match Report	
Group stage	4	Thu	2021-11-04	18:45	AZ Alkmaar nl	2–0	ro CFR Cluj	10,554	AFAS Stadion	Alain Durieux	Match Report	
Group stage	4	Thu	2021-11-04	18:45	Lincoln Red Imps FC gi	1–4	sk Slovan Bratislava	553	Victoria Stadium	Kristoffer Karlsson	Match Report	
Group stage	4	Thu	2021-11-04	18:45	Gent be	1–1	rs Partizan	10,595	GHELAMCO-arena	Chris Kavanagh	Match Report	
Group stage	4	Thu	2021-11-04	18:45	Randers dk	2–2	cz Jablonec	4,071	Cepheus Park Randers	Orel Grinfeeld	Match Report	
Group stage	4	Thu	2021-11-04	19:45 (18:45)	Zorya Luhansk ua	2–0	bg CSKA Sofia	1,122	Slavutych-Arena	Petri Viljanen	Match Report	
Group stage	4	Thu	2021-11-04	19:45 (18:45)	AC Omonia cy	1–1	ch Basel	4,597	Neo GSP	Vitaly Meshkov	Match Report	
Group stage	4	Thu	2021-11-04	19:45 (18:45)	Maccabi Tel Aviv FC il	3–0	fi HJK Helsinki	8,029	Bloomfield Stadium	Rade Obrenović	Match Report	
Group stage	4	Thu	2021-11-04	20:00 (21:00)	Tottenham Hotspur eng	3–2	nl Vitesse	36,312	Tottenham Hotspur Stadium	Marco Di Bello	Match Report	
Group stage	4	Thu	2021-11-04	21:00	Union Berlin de	1–2	nl Feyenoord	30,000	Olympiastadion Berlin	Daniel Stefański	Match Report	
Group stage	4	Thu	2021-11-04	21:00	Rennes fr	1–0	si NŠ Mura	23,115	Roazhon Park	Juri Frischer	Match Report	
Group stage	4	Thu	2021-11-04	21:00	Roma it	2–2	no Bodø/Glimt	41,031	Stadio Olimpico	Anastasios Papapetrou	Match Report	
Group stage	4	Thu	2021-11-04	21:00	Slavia Prague cz	1–0	il Maccabi Haifa	13,646	Sinobo Stadium	Kai Erik Steen	Match Report	
Group stage	4	Thu	2021-11-04	21:30 (16:30)	FC Kairat kz	1–2	az Qarabağ	2,638	Ortalıq Stadion	Dumitru Muntean	Match Report	
Group stage	4	Thu	2021-11-04	22:00 (21:00)	PAOK gr	1–2	dk FC Copenhagen	11,166	Stadio Toumbas	Paweł Raczkowski	Match Report	
Group stage	5	Thu	2021-11-25	17:30 (16:30)	FC Flora ee	1–0	rs Partizan	1,503	A. Le Coq Arena	Robert Harvey	Match Report	
Group stage	5	Thu	2021-11-25	18:45	Slavia Prague cz	2–2	nl Feyenoord	14,562	Sinobo Stadium	Mattias Gestranius	Match Report	
Group stage	5	Thu	2021-11-25	18:45	NŠ Mura si	2–1	eng Tottenham Hotspur	6,100	Ljudski vrt	António Nobre	Match Report	
Group stage	5	Thu	2021-11-25	18:45	Rennes fr	3–3	nl Vitesse	23,081	Roazhon Park	Espen Andreas Eskås	Match Report	
Group stage	5	Thu	2021-11-25	18:45	Slovan Bratislava sk	0–0	gr PAOK	200	Štadión Tehelné pole	Dennis Higler	Match Report	
Group stage	5	Thu	2021-11-25	18:45	Lincoln Red Imps FC gi	0–4	dk FC Copenhagen	1,008	Victoria Stadium	Helgi Mikael Jónasson	Match Report	
Group stage	5	Thu	2021-11-25	19:45 (18:45)	Maccabi Haifa il	0–1	de Union Berlin	22,150	Sammy Ofer Stadium	Michal Očenáš	Match Report	
Group stage	5	Thu	2021-11-25	19:45 (18:45)	HJK Helsinki fi	1–0	am Alashkert	4,424	Telia 5G -areena	Bram Van Driessche	Match Report	
Group stage	5	Thu	2021-11-25	21:00	Bodø/Glimt no	2–0	bg CSKA Sofia	5,339	Aspmyra Stadion	Igal Frid	Match Report	
Group stage	5	Thu	2021-11-25	21:00	Roma it	4–0	ua Zorya Luhansk	24,000	Stadio Olimpico	István Kovács	Match Report	
Group stage	5	Thu	2021-11-25	21:00	Randers dk	2–1	ro CFR Cluj	4,309	Cepheus Park Randers	Donald Robertson	Match Report	
Group stage	5	Thu	2021-11-25	21:00	Jablonec cz	1–1	nl AZ Alkmaar	2,650	Stadion Střelnice	Manuel Schuettengruber	Match Report	
Group stage	5	Thu	2021-11-25	21:30 (16:30)	FC Kairat kz	2–3	ch Basel	4,159	Ortalıq Stadion	Goga Kikacheishvili	Match Report	
Group stage	5	Thu	2021-11-25	21:45 (18:45)	Qarabağ az	2–2	cy AC Omonia	17,231	Tofiq Bəhramov adına Respublika stadionu	Morten Krogh	Match Report	
Group stage	5	Thu	2021-11-25	22:00 (21:00)	Anorthosis cy	1–0	be Gent	2,573	Neo GSP	Urs Schnyder	Match Report	
Group stage	5	Thu	2021-11-25	22:00 (21:00)	Maccabi Tel Aviv FC il	0–1	at LASK	12,203	Bloomfield Stadium	Jérôme Brisard	Match Report	
Group stage	6	Thu	2021-12-09	18:45	Gent be	1–0	ee FC Flora	7,421	GHELAMCO-arena	Kaspar Sjöberg	Match Report	
Group stage	6	Thu	2021-12-09	18:45	LASK at	3–0	fi HJK Helsinki		Wörthersee Stadion	Igor Pajač	Match Report	
Group stage	6	Thu	2021-12-09	18:45	Partizan rs	1–1	cy Anorthosis	4,493	Stadion Partizana	Giorgi Kruashvili	Match Report	
Group stage	6	Thu	2021-12-09	18:45	AZ Alkmaar nl	1–0	dk Randers		AFAS Stadion	Trustin Farrugia Cann	Match Report	
Group stage	6	Thu	2021-12-09	19:45 (18:45)	CFR Cluj ro	2–0	cz Jablonec	2,400	Stadionul Dr. Constantin Rădulescu	Harm Osmers	Match Report	
Group stage	6	Thu	2021-12-09	19:45 (18:45)	CSKA Sofia bg	2–3	it Roma	4,640	Stadion Vasil Levski	Nick Walsh	Match Report	
Group stage	6	Thu	2021-12-09	19:45 (18:45)	Zorya Luhansk ua	1–1	no Bodø/Glimt	1,134	Slavutych-Arena	Krzysztof Jakubik	Match Report	
Group stage	6	Thu	2021-12-09	20:00 (21:00)	Tottenham Hotspur eng	0–3	fr Rennes		Tottenham Hotspur Stadium		Match Report	Match awarded to Rennes
Group stage	6	Thu	2021-12-09	21:00	FC Copenhagen dk	2–0	sk Slovan Bratislava	13,021	Parken	István Vad	Match Report	
Group stage	6	Thu	2021-12-09	21:00	Union Berlin de	1–1	cz Slavia Prague	4,380	Olympiastadion Berlin	Rade Obrenović	Match Report	
Group stage	6	Thu	2021-12-09	21:00	Feyenoord nl	2–1	il Maccabi Haifa		Stadion Feijenoord	Mykola Balakin	Match Report	
Group stage	6	Thu	2021-12-09	21:00	Vitesse nl	3–1	si NŠ Mura		GelreDome	Filip Glova	Match Report	
Group stage	6	Thu	2021-12-09	21:00	Basel ch	3–0	az Qarabağ	10,059	St. Jakob-Park	Fran Jović	Match Report	
Group stage	6	Thu	2021-12-09	21:45 (18:45)	Alashkert am	1–1	il Maccabi Tel Aviv FC	700	Vazgen Sargsyan anvan Hanrapetakan Marza...	Gergő Bogár	Match Report	
Group stage	6	Thu	2021-12-09	22:00 (21:00)	PAOK gr	2–0	gi Lincoln Red Imps FC	4,648	Stadio Toumbas	Rohit Saggi	Match Report	
Group stage	6	Thu	2021-12-09	22:00 (21:00)	AC Omonia cy	0–0	kz FC Kairat	3,239	Neo GSP	Genc Nuza	Match Report	
Knockout round play-offs		Thu	2022-02-17	18:45	Midtjylland dk	1–0	gr PAOK	7,088	MCH Arena	Craig Pawson	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	18:45	PSV nl	1–0	il Maccabi Tel Aviv FC	14,280	Philips Stadion	Glenn Nyberg	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	18:45	Rapid Wien at	2–1	nl Vitesse	10,700	Allianz Stadion	Donatas Rumšas	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	20:00 (21:00)	Leicester City eng	4–1	dk Randers	25,242	King Power Stadium	Radu Petrescu	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	20:00 (21:00)	Celtic sct	1–3	no Bodø/Glimt	54,926	Celtic Park	Andris Treimanis	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	20:45 (18:45)	Fenerbahçe tr	2–3	cz Slavia Prague	30,094	Ülker Stadyumu Fenerbahçe Şükrü Saracoğl...	Maurizio Mariani	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	21:00	Marseille fr	3–1	az Qarabağ	25,173	Orange Vélodrome	Nikola Dabanović	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-17	21:00	Sparta Prague cz	0–1	rs Partizan	1,000	Generali Česká Pojišťovna Arena	Andreas Ekberg	Match Report	Leg 1 of 2
Knockout round play-offs		Thu	2022-02-24	18:45	Randers dk	1–3	eng Leicester City	8,948	Cepheus Park Randers	Irfan Peljto	Match Report	Leg 2 of 2; Leicester City won
Knockout round play-offs		Thu	2022-02-24	18:45	Partizan rs	2–1	cz Sparta Prague	13,864	Stadion Partizana	Georgi Kabakov	Match Report	Leg 2 of 2; Partizan won
Knockout round play-offs		Thu	2022-02-24	18:45	Bodø/Glimt no	2–0	sct Celtic	5,801	Aspmyra Stadion	Sergey Ivanov	Match Report	Leg 2 of 2; Bodø/Glimt won
Knockout round play-offs		Thu	2022-02-24	19:45 (18:45)	Maccabi Tel Aviv FC il	1–1	nl PSV	28,058	Bloomfield Stadium	Harald Lechner	Match Report	Leg 2 of 2; PSV won
Knockout round play-offs		Thu	2022-02-24	21:00	Vitesse nl	2–0	at Rapid Wien	13,517	GelreDome	Jakob Kehlet	Match Report	Leg 2 of 2; Vitesse won
Knockout round play-offs		Thu	2022-02-24	21:00	Slavia Prague cz	3–2	tr Fenerbahçe	10,120	Sinobo Stadium	Chris Kavanagh	Match Report	Leg 2 of 2; Slavia Prague won
Knockout round play-offs		Thu	2022-02-24	21:45 (18:45)	Qarabağ az	0–3	fr Marseille	27,783	Tofiq Bəhramov adına Respublika stadionu	Bartosz Frankowski	Match Report	Leg 2 of 2; Marseille won
Knockout round play-offs		Thu	2022-02-24	22:00 (21:00)	PAOK gr	(5) 2–1 (3)	dk Midtjylland	7,468	Stadio Toumbas	Lawrence Visser	Match Report	Leg 2 of 2; PAOK won; PAOK won on penalty kicks following extra time
Round of 16		Thu	2022-03-10	18:45	Slavia Prague cz	4–1	at LASK	16,754	Sinobo Stadium	Andris Treimanis	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	18:45	Partizan rs	2–5	nl Feyenoord	13,564	Stadion Partizana	Maurizio Mariani	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	18:45	Vitesse nl	0–1	it Roma	24,022	GelreDome	Paweł Raczkowski	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	19:45 (18:45)	PAOK gr	1–0	be Gent	9,098	Stadio Toumbas	Bartosz Frankowski	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	20:00 (21:00)	Leicester City eng	2–0	fr Rennes	25,848	The Leicester City Training Ground	Orel Grinfeeld	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	21:00	Marseille fr	2–1	ch Basel	22,992	Orange Vélodrome	Irfan Peljto	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	21:00	Bodø/Glimt no	2–1	nl AZ Alkmaar	5,724	Aspmyra Stadion	Nikola Dabanović	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-10	21:00	PSV nl	4–4	dk FC Copenhagen	26,003	Philips Stadion	Craig Pawson	Match Report	Leg 1 of 2
Round of 16		Thu	2022-03-17	18:45	AZ Alkmaar nl	2–2	no Bodø/Glimt	15,024	AFAS Stadion	Ivan Kružliak	Match Report	Leg 2 of 2; Bodø/Glimt won; Required Extra Time
Round	Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
Round of 16		Thu	2022-03-17	18:45	FC Copenhagen dk	0–4	nl PSV	30,337	Parken	William Collum	Match Report	Leg 2 of 2; PSV won
Round of 16		Thu	2022-03-17	18:45	Basel ch	1–2	fr Marseille	22,081	St. Jakob-Park	Artur Dias	Match Report	Leg 2 of 2; Marseille won
Round of 16		Thu	2022-03-17	18:45	Rennes fr	2–1	eng Leicester City	27,000	Roazhon Park	Anastasios Sidiropoulos	Match Report	Leg 2 of 2; Leicester City won
Round of 16		Thu	2022-03-17	21:00	Feyenoord nl	3–1	rs Partizan	40,850	Stadion Feijenoord	Anthony Taylor	Match Report	Leg 2 of 2; Feyenoord won
Round of 16		Thu	2022-03-17	21:00	LASK at	4–3	cz Slavia Prague	5,403	NV Arena	Lawrence Visser	Match Report	Leg 2 of 2; Slavia Prague won
Round of 16		Thu	2022-03-17	21:00	Roma it	1–1	nl Vitesse	40,435	Stadio Olimpico	Radu Petrescu	Match Report	Leg 2 of 2; Roma won
Round of 16		Thu	2022-03-17	21:00	Gent be	1–2	gr PAOK	12,388	GHELAMCO-arena	Daniel Siebert	Match Report	Leg 2 of 2; PAOK won
Quarter-finals		Thu	2022-04-07	18:45	Feyenoord nl	3–3	cz Slavia Prague	42,163	Stadion Feijenoord	Halil Umut Meler	Match Report	Leg 1 of 2
Quarter-finals		Thu	2022-04-07	20:00 (21:00)	Leicester City eng	0–0	nl PSV	31,327	King Power Stadium	Ivan Kružliak	Match Report	Leg 1 of 2
Quarter-finals		Thu	2022-04-07	21:00	Bodø/Glimt no	2–1	it Roma	7,739	Aspmyra Stadion	Serdar Gözübüyük	Match Report	Leg 1 of 2
Quarter-finals		Thu	2022-04-07	21:00	Marseille fr	2–1	gr PAOK	45,182	Orange Vélodrome	Szymon Marciniak	Match Report	Leg 1 of 2
Quarter-finals		Thu	2022-04-14	18:45	PSV nl	1–2	eng Leicester City	35,000	Philips Stadion	Benoît Bastien	Match Report	Leg 2 of 2; Leicester City won
Quarter-finals		Thu	2022-04-14	21:00	Slavia Prague cz	1–3	nl Feyenoord	19,370	Sinobo Stadium	Deniz Aytekin	Match Report	Leg 2 of 2; Feyenoord won
Quarter-finals		Thu	2022-04-14	21:00	Roma it	4–0	no Bodø/Glimt	61,942	Stadio Olimpico	José Sánchez	Match Report	Leg 2 of 2; Roma won
Quarter-finals		Thu	2022-04-14	22:00 (21:00)	PAOK gr	0–1	fr Marseille	27,648	Stadio Toumbas	Danny Makkelie	Match Report	Leg 2 of 2; Marseille won
Semi-finals		Thu	2022-04-28	20:00 (21:00)	Leicester City eng	1–1	it Roma	31,659	The Leicester City Training Ground	Carlos del Cerro	Match Report	Leg 1 of 2
Semi-finals		Thu	2022-04-28	21:00	Feyenoord nl	3–2	fr Marseille	24,500	Stadion Feijenoord	Michael Oliver	Match Report	Leg 1 of 2
Semi-finals		Thu	2022-05-05	21:00	Roma it	1–0	eng Leicester City	63,940	Stadio Olimpico	Srđan Jovanović	Match Report	Leg 2 of 2; Roma won
Semi-finals		Thu	2022-05-05	21:00	Marseille fr	0–0	nl Feyenoord	49,315	Orange Vélodrome	Sandro Schärer	Match Report	Leg 2 of 2; Feyenoord won
Final		Wed	2022-05-25	21:00	Roma it	1–0	nl Feyenoord	1

"""

In [220]:
def strip_country_code(name):
    """Remove 2-3 letter country codes FBRef adds to team names in European comps.
    e.g. 'eng Arsenal' -> 'Arsenal', 'Tottenham Hotspur eng' -> 'Tottenham Hotspur'
    """
    import re
    name = name.strip()
    name = re.sub(r'^[a-z]{2,3}\s+', '', name)  # code at start
    name = re.sub(r'\s+[a-z]{2,3}$', '', name)  # code at end
    return name.strip()


def parse_fbref_paste(raw_text, competition, season):
    rows = []
    has_round_col = False

    for line in raw_text.strip().splitlines():
        parts = line.split('\t')
        if len(parts) < 7:
            continue

        # Detect format from header row
        if parts[0].strip() == 'Round':
            has_round_col = True
            continue
        if parts[0].strip() in ('Wk', ''):
            continue

        # Shift indices if there's an extra Round column
        if has_round_col:
            # Round Wk Day Date Time Home Score Away Attendance Venue ...
            round_  = parts[0].strip()
            date_str  = parts[3].strip()
            home_team = parts[5].strip()
            score     = parts[6].strip()
            away_team = parts[7].strip()
            venue     = parts[9].strip() if len(parts) > 9 else ''
        else:
            # Wk Day Date Time Home Score Away Attendance Venue ...
            round_    = parts[0].strip()
            date_str  = parts[2].strip()
            home_team = parts[4].strip()
            score     = parts[5].strip()
            away_team = parts[6].strip()
            venue     = parts[8].strip() if len(parts) > 8 else ''

        # Strip country codes from European competition team names
        home_team = strip_country_code(home_team)
        away_team = strip_country_code(away_team)

        # Skip rows with no score
        if not any(c in score for c in ['\u2013', '-']):
            continue
        if not date_str or not home_team or not away_team:
            continue

        # Parse score
        home_goals = away_goals = None
        for sep in ['\u2013', '-']:
            if sep in score:
                s = score.split(sep)
                try:
                    home_goals = int(s[0].strip())
                    away_goals = int(s[1].strip())
                    break
                except (ValueError, IndexError):
                    pass

        if home_goals is None:
            continue

        rows.append({
            'date':        date_str,
            'season':      season,
            'league_name': competition,
            'round':       round_,
            'home_team':   home_team,
            'away_team':   away_team,
            'home_goals':  home_goals,
            'away_goals':  away_goals,
            'venue':       venue,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    return df


parsed = parse_fbref_paste(RAW_TEXT, COMPETITION, SEASON)
print(f'Parsed {len(parsed)} fixtures for {COMPETITION} {SEASON}')
parsed.head(10)


Parsed 140 fixtures for Conference League 2021-2022


,date,season,league_name,round,home_team,away_team,home_goals,away_goals,venue
0,2021-09-14,2021-2022,Conference League,Group stage,Maccabi Tel Aviv FC,Alashkert,4,1,Bloomfield Stadium
1,2021-09-14,2021-2022,Conference League,Group stage,Maccabi Haifa,Feyenoord,0,0,Sammy Ofer Stadium
2,2021-09-16,2021-2022,Conference League,Group stage,Anorthosis,Partizan,0,2,Stadio Antonis Papadopoulos
3,2021-09-16,2021-2022,Conference League,Group stage,Randers,AZ Alkmaar,2,2,Cepheus Park Randers
4,2021-09-16,2021-2022,Conference League,Group stage,Roma,CSKA Sofia,5,1,Stadio Olimpico
5,2021-09-16,2021-2022,Conference League,Group stage,Bodø/Glimt,Zorya Luhansk,3,1,Aspmyra Stadion
6,2021-09-16,2021-2022,Conference League,Group stage,Qarabağ,Basel,0,0,Tofiq Bəhramov adına Respublika stadionu
7,2021-09-16,2021-2022,Conference League,Group stage,FC Kairat,AC Omonia,0,0,Ortalıq Stadion
8,2021-09-16,2021-2022,Conference League,Group stage,Jablonec,CFR Cluj,1,0,Stadion Střelnice
9,2021-09-16,2021-2022,Conference League,Group stage,HJK Helsinki,LASK,0,2,Telia 5G -areena


In [221]:
# Append to master CSV (deduplicates on date + home_team + away_team)
if parsed.empty:
    print("Nothing to save — check the paste and re-run")
else:
    if MASTER_CSV.exists():
        existing = pd.read_csv(MASTER_CSV, parse_dates=["date"])
        combined = pd.concat([existing, parsed], ignore_index=True)
        combined = combined.drop_duplicates(subset=["date", "home_team", "away_team"])
    else:
        combined = parsed

    combined = combined.sort_values(["season", "league_name", "date"]).reset_index(drop=True)
    combined.to_csv(MASTER_CSV, index=False)
    print(f"Saved {len(combined)} total rows → {MASTER_CSV}")
    print()
    print(combined.groupby(["league_name", "season"]).size().unstack(fill_value=0).to_string())

Saved 8626 total rows → /Users/admin/dev/algobetting/infra/data/json/fbref/fixtures_manual.csv

season             2017-2018  2018-2019  2019-2020  2020-2021  2021-2022  2022-2023  2023-2024  2024-2025  2025-2026
league_name                                                                                                         
Champions League         125        125          0        125        125        125        122        187        184
Conference League          0          0          0          0        140        137        138        151        148
EFL Cup                   89         73         69         71         62         75         69         78         73
Europa League            205        204        197        204        137        137        139        187        182
FA Cup                   147        145        153        112        133        141        145        105        103
Premier League           380        380        380        380        380        380  